# DUNE/FABRIC one-second incremental viewer

This viewer requests only complete CSV rows appended since the previous request. Set `URL` to the local port printed by `loomai tunnels open`, then run the cell before starting capture.

In [ ]:
import io
import time
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import requests
from IPython.display import clear_output, display

URL = "http://127.0.0.1:9104/"  # Change to the tunnel's local port
EXPECTED_SAMPLES = 409_600
POLL_SECONDS = 0.5
MAX_BYTES = 262_144
CSV_FIELDS = [
    "seq_id", "timestamp", "tick", "adc_value",
    "wire_id", "tpc", "slot", "plane", "packet_file",
]

offset = 0
values = np.full((200, 2048), np.nan, dtype=float)
history = []
last_status = None

for request_number in range(3600):
    try:
        request_started = time.perf_counter()
        response = requests.get(
            URL,
            params={"offset": offset, "max_bytes": MAX_BYTES},
            headers={"X-Byte-Offset": str(offset)},
            timeout=5,
        )
        request_seconds = time.perf_counter() - request_started

        if response.status_code == 204:
            status = f"Waiting for new rows at byte offset {offset:,}"
            if status != last_status:
                clear_output(wait=True)
                print(status)
                last_status = status
            time.sleep(POLL_SECONDS)
            continue

        response.raise_for_status()
        content_type = response.headers.get("Content-Type", "")
        if not content_type.startswith("text/csv"):
            raise RuntimeError(
                f"Expected CSV but received {content_type}: "
                f"{response.text[:160]}"
            )

        stream_reset = response.headers.get("X-Stream-Reset") == "1"
        has_header = offset == 0 or stream_reset
        if stream_reset:
            values[:] = np.nan
            history.clear()

        if has_header:
            batch = pd.read_csv(io.BytesIO(response.content))
        else:
            batch = pd.read_csv(
                io.BytesIO(response.content),
                header=None,
                names=CSV_FIELDS,
            )

        offset = int(response.headers["X-Next-Offset"])
        batch = batch.dropna(
            subset=["tick", "adc_value", "wire_id", "tpc", "plane"]
        )
        batch = batch[(batch["tpc"] == 0) & (batch["plane"] == 0)]
        batch = batch[
            batch["wire_id"].between(0, 199)
            & batch["tick"].between(0, 2047)
        ]
        wires = batch["wire_id"].astype(int).to_numpy()
        ticks = batch["tick"].astype(int).to_numpy()
        values[wires, ticks] = batch["adc_value"].to_numpy(dtype=float)

        sample_count = int(np.isfinite(values).sum())
        completion = min(100, sample_count / EXPECTED_SAMPLES * 100)
        display_values = values.copy()
        for row in range(display_values.shape[0]):
            valid = np.isfinite(display_values[row])
            if valid.any():
                display_values[row, valid] -= np.median(display_values[row, valid])

        file_mtime_ns = int(response.headers.get("X-File-Mtime-Ns", "0"))
        display_lag = (time.time_ns() - file_mtime_ns) / 1_000_000_000
        history.append({
            "time": time.strftime("%H:%M:%S"),
            "samples": sample_count,
            "completion_%": round(completion, 1),
            "new_rows": len(batch),
            "request_ms": round(request_seconds * 1000, 1),
            "display_lag_s": round(max(0, display_lag), 2),
        })

        figure = go.Figure(go.Heatmap(
            z=display_values,
            x=np.arange(2048),
            y=np.arange(200),
            colorscale="Thermal",
            zmin=-20,
            zmax=50,
            colorbar={"title": "ADC - pedestal"},
        ))
        figure.update_layout(
            title=(
                f"DUNE/FABRIC live replay - {sample_count:,} samples "
                f"({completion:.1f}%)"
            ),
            xaxis_title="Time tick",
            yaxis_title="Wire ID",
            template="plotly_dark",
            height=750,
        )
        figure.update_xaxes(range=[0, 2047])
        figure.update_yaxes(range=[199, 0])
        clear_output(wait=True)
        display(figure)
        display(pd.DataFrame(history))
        last_status = None

        if sample_count >= EXPECTED_SAMPLES:
            print("Replay complete.")
            break
    except Exception as error:
        clear_output(wait=True)
        print("Waiting for live delta service:", error)

    time.sleep(POLL_SECONDS)
